# 確率的エラー増幅によるユーティリティー・スケールのエラー緩和

*使用量の目安: Heron r3 プロセッサで 14 分（注意: これはあくまで目安です。実際の実行時間は異なる場合があります。）*

## 学習成果

* *ゼロノイズ外挿*（ZNE）の背後にある理論、ノイズを増幅するさまざまな方法、そしてユーティリティー・スケールの実験で *確率的エラー増幅*（PEA）が好まれる理由
* Qiskit を使って、PEA による ZNE を実際に実装する方法

## 前提知識

* Qiskit でエラー緩和を使うための基礎知識として、*ユーティリティー・スケールの量子コンピューティング* コースの [エラー緩和のレッスン](/learning/courses/utility-scale-quantum-computing/error-mitigation)
* このチュートリアルで例として使うユーティリティー・スケールの実験の背景として、*ユーティリティー・スケールの量子コンピューティング* コースの [Utility-I のレッスン](/learning/courses/utility-scale-quantum-computing/utility-i)

## 背景

このチュートリアルでは、*確率的エラー増幅*（PEA）を用いた *ゼロノイズ外挿*（ZNE）の実験的なバージョンを使って、IBM Quantum Compute Service でユーティリティー・スケールのエラー緩和実験を実行する方法を示します。

![kim\_nature\_fig.png](https://quantum.cloud.ibm.com/docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/e1e67c34-9d4d-4a88-9340-f0b2f3676770.avif)
**参考文献**: Y. Kim et al. *Evidence for the utility of quantum computing before fault tolerance.* [Nature 618.7965 (2023)](https://www.nature.com/articles/s41586-023-06096-3)

### ゼロノイズ外挿（ZNE）

ゼロノイズ外挿（ZNE）は、回路の実行中に生じる *未知の* ノイズのうち、*既知の* 方法でスケールできるものの影響を取り除くエラー緩和手法です。

ZNE では、期待値がノイズに対して既知の関数に従ってスケールすると仮定します。

$$
\langle A(\lambda) \rangle = \langle A(0) \rangle + \sum_{k=0}^{m} a_k \lambda^k + R
$$

ここで、$\lambda$ はノイズの強さを表すパラメーターであり、増幅することができます。

ZNE は次の手順で実装できます。

1. 複数のノイズ係数 $\lambda_1, \lambda_2, \ldots$ について回路のノイズを増幅する
2. ノイズを増幅した各回路を実行して $\langle A(\lambda_1)\rangle, \ldots$ を測定する
3. ノイズゼロの極限 $\langle A(0)\rangle$ へ外挿する

![zne\_stages.png](https://quantum.cloud.ibm.com/docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/5e63d706-82d8-4212-b802-c9191ce53341.avif)

#### ZNE のためのノイズ増幅

ZNE をうまく実装するうえでの主な課題は、期待値に含まれるノイズの正確なモデルを持ち、そのノイズを既知の方法で増幅することです。

ZNE のためのエラー増幅は、一般に次の 3 つの方法で実装されます。

| **パルス・ストレッチング** | **ゲート・フォールディング** | **確率的エラー増幅** |
| -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| キャリブレーションによってパルスの時間幅をスケールする | 恒等サイクルでゲートを繰り返す $U\mapsto U(U^{-1}U)^{\lambda-1}/2$ | パウリ・チャネルのサンプリングによってノイズを加える |
| ![zne\_pulse\_stretching.png](https://quantum.cloud.ibm.com/docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/83188b57-e88f-43a1-a7bd-29327f46ecf5.avif) | ![zne\_gate\_folding.png](https://quantum.cloud.ibm.com/docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/e1358d08-2632-4fd2-bf0f-f9384a2d3340.avif) | ![zne\_pea.png](https://quantum.cloud.ibm.com/docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/3d69d5bd-70e5-4eeb-aa02-fc0a62043010.avif) |
| Kandala et al. Nature (2019)                                                                                                                                                                       | Shultz et al. PRA (2022)                                                                                                                                                                       | Li & Benjamin PRX (2017)                                                                                                                                                             |



ユーティリティー・スケールの実験では、*確率的エラー増幅*（PEA）が最も魅力的です。

* パルス・ストレッチングは、ゲートのノイズがパルスの時間幅に比例すると仮定しますが、通常この仮定は成り立ちません。また、キャリブレーションのコストも高くなります。
* ゲート・フォールディングでは大きな引き伸ばし係数が必要になるため、実行できる回路の深さが大きく制限されます。
* PEA は、本来のノイズ係数（$\lambda=1$）で実行できる回路であればどのような回路にも適用できますが、ノイズモデルを学習する必要があります。

### PEA のためのノイズモデルの学習

PEA は、*確率的エラー・キャンセレーション*（PEC）と同じ層ベースのノイズモデルを仮定しますが、回路のノイズに対して指数関数的に増加するサンプリング・オーバーヘッドを回避できます。

| **ステップ 1** | **ステップ 2** | **ステップ 3** |
| --- | --- | --- |
| 2 量子ビット・ゲートの層をパウリ・ツワーリングする | 恒等演算となる層のペアを繰り返し、ノイズを学習する | 忠実度（各ノイズチャネルのエラー）を導出する |
| ![pec\_pauli\_twirling.png](https://quantum.cloud.ibm.com/docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/2eab5ff4-40fa-4a41-9f2c-74f5e22c4643.avif) | ![pec\_learn\_layer.png](https://quantum.cloud.ibm.com/docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/8d0d64c3-65ad-4419-8ac9-4ec9633d39a0.avif) | ![pec\_curve\_fitting.png](https://quantum.cloud.ibm.com/docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/c51bd42d-2463-4c78-807b-d284ca79296f.avif) |

**参考文献**: E. van den Berg, Z. Minev, A. Kandala, and K. Temme, *Probabilistic error cancellation with sparse Pauli-Lindblad models on noisy quantum processors* [arXiv:2201.09866](https://arxiv.org/abs/2201.09866)

## 要件

このチュートリアルを始める前に、次のものがインストールされていることを確認してください。

* Qiskit SDK v2.0 以降（[可視化](/docs/api/qiskit/visualization) サポート付き）
* Qiskit Runtime v0.22 以降（`pip install qiskit-ibm-runtime`）

## セットアップ

次のセルでは、関連するパッケージをインポートし、バックエンドのトポロジーに沿った 2 次元横磁場イジング模型のトロッター化された時間発展の回路を構成するためのヘルパー関数をいくつか作成します。

In [1]:
from __future__ import annotations
from collections.abc import Sequence
from collections import defaultdict
import numpy as np
import rustworkx
import matplotlib.pyplot as plt

from qiskit.circuit import QuantumCircuit, Parameter
from qiskit.circuit.library import CXGate, CZGate, ECRGate
from qiskit.providers import Backend
from qiskit.visualization import plot_error_map
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import PubResult

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator


"""Trotter circuit generation"""


def remove_qubit_couplings(
    couplings: Sequence[tuple[int, int]], qubits: Sequence[int] | None = None
) -> list[tuple[int, int]]:
    """Remove qubits from a coupling list.

    Args:
        couplings: A sequence of qubit couplings.
        qubits: Optional, the qubits to remove.

    Returns:
        The input couplings with the specified qubits removed.
    """
    if qubits is None:
        return couplings
    qubits = set(qubits)
    return [edge for edge in couplings if not qubits.intersection(edge)]


def coupling_qubits(
    *couplings: Sequence[tuple[int, int]],
    allowed_qubits: Sequence[int] | None = None,
) -> list[int]:
    """Return a sorted list of all qubits involved in one or more couplings lists.

    Args:
        couplings: one or more coupling lists.
        allowed_qubits: Optional, the allowed qubits to include. If None all
            qubits are allowed.

    Returns:
        The intersection of all qubits in the couplings and the allowed qubits.
    """
    qubits = set()
    for edges in couplings:
        for edge in edges:
            qubits.update(edge)
    if allowed_qubits is not None:
        qubits = qubits.intersection(allowed_qubits)
    return list(qubits)


def construct_layer_couplings(
    backend: Backend,
) -> list[list[tuple[int, int]]]:
    """Separate a coupling map into disjoint 2-qubit gate layers.

    Args:
        backend: A backend to construct layer couplings for.

    Returns:
        A list of disjoint layers of directed couplings for the input coupling map.
    """
    coupling_graph = backend.coupling_map.graph.to_undirected(
        multigraph=False
    )
    edge_coloring = rustworkx.graph_bipartite_edge_color(coupling_graph)

    layers = defaultdict(list)
    for edge_idx, color in edge_coloring.items():
        layers[color].append(
            coupling_graph.get_edge_endpoints_by_index(edge_idx)
        )
    layers = [sorted(layers[i]) for i in sorted(layers.keys())]

    return layers


def entangling_layer(
    gate_2q: str,
    couplings: Sequence[tuple[int, int]],
    qubits: Sequence[int] | None = None,
) -> QuantumCircuit:
    """Generating a entangling layer for the specified couplings.

    This corresponds to a Trotter layer for a ZZ Ising term with angle Pi/2.

    Args:
        gate_2q: The 2-qubit basis gate for the layer, should be "cx", "cz", or "ecr".
        couplings: A sequence of qubit couplings to add CX gates to.
        qubits: Optional, the physical qubits for the layer. Any couplings involving
            qubits not in this list will be removed. If None the range up to the largest
            qubit in the couplings will be used.

    Returns:
        The QuantumCircuit for the entangling layer.
    """
    # 量子ビットを取得し、順序付けのために集合に変換する
    if qubits is None:
        qubits = range(1 + max(coupling_qubits(couplings)))
    qubits = set(qubits)

    # 物理量子ビットから仮想量子ビットへの対応付け
    qubit_mapping = {q: i for i, q in enumerate(qubits)}

    # カップリングを仮想量子ビットのインデックスに変換する
    indices = [
        [qubit_mapping[i] for i in edge]
        for edge in couplings
        if qubits.issuperset(edge)
    ]

    # 仮想量子ビット上の層の回路
    circuit = QuantumCircuit(len(qubits))

    # 2 量子ビットの基底ゲートと、前後の回転の回路を取得する
    gate2q = None
    pre = QuantumCircuit(2)
    post = QuantumCircuit(2)

    if gate_2q == "cx":
        gate2q = CXGate()
        # 前回転
        pre.sdg(0)
        pre.z(1)
        pre.sx(1)
        pre.s(1)
        # 後回転
        post.sdg(1)
        post.sxdg(1)
        post.s(1)
    elif gate_2q == "ecr":
        gate2q = ECRGate()
        # 前回転
        pre.z(0)
        pre.s(1)
        pre.sx(1)
        pre.s(1)
        # 後回転
        post.x(0)
        post.sdg(1)
        post.sxdg(1)
        post.s(1)
    elif gate_2q == "cz":
        gate2q = CZGate()
        # 前回転は恒等演算
        # 後回転
        post.sdg([0, 1])
    else:
        raise ValueError(
            f"Invalid 2-qubit basis gate {gate_2q}, should be 'cx', 'cz', or 'ecr'"
        )

    # 1 量子ビットの前回転を追加する
    for inds in indices:
        circuit.compose(pre, qubits=inds, inplace=True)

    # 2 量子ビットの基底ゲートをバリアで囲み、PEA のノイズ学習の対象となる層を指定する
    circuit.barrier()
    for inds in indices:
        circuit.append(gate2q, (inds[0], inds[1]))
    circuit.barrier()

    # バリアの後に 1 量子ビットの後回転を追加する
    for inds in indices:
        circuit.compose(post, qubits=inds, inplace=True)

    # 物理量子ビットをメタデータとして追加する
    circuit.metadata["physical_qubits"] = tuple(qubits)

    return circuit


def trotter_circuit(
    theta: Parameter | float,
    layer_couplings: Sequence[Sequence[tuple[int, int]]],
    num_steps: int,
    gate_2q: str | None = "cx",
    backend: Backend | None = None,
    qubits: Sequence[int] | None = None,
) -> QuantumCircuit:
    """Generate a Trotter circuit for the 2D Ising

    Args:
        theta: The angle parameter for X.
        layer_couplings: A list of couplings for each entangling layer.
        num_steps: the number of Trotter steps.
        gate_2q: The 2-qubit basis gate to use in entangling layers.
            Can be "cx", "cz", "ecr", or None if a backend is provided.
        backend: A backend to get the 2-qubit basis gate from, if provided
            will override the basis_gate field.
        qubits: Optional, the allowed physical qubits to truncate the
            couplings to. If None the range up to the largest
            qubit in the couplings will be used.

    Returns:
        The Trotter circuit.
    """
    if backend is not None:
        try:
            basis_gates = backend.configuration().basis_gates
        except AttributeError:
            basis_gates = backend.basis_gates
        for gate in ["cx", "cz", "ecr"]:
            if gate in basis_gates:
                gate_2q = gate
                break

    # 量子ビットが指定されていない場合は、すべての層から最大の量子ビットを取得し、
    # すべての層で同じ範囲が使われるように範囲を指定する。
    if qubits is None:
        qubits = range(1 + max(coupling_qubits(layer_couplings)))

    # エンタングリング層を生成する
    layers = [
        entangling_layer(gate_2q, couplings, qubits=qubits)
        for couplings in layer_couplings
    ]

    # 1 トロッター・ステップ分の回路を構成する
    num_qubits = len(qubits)
    trotter_step = QuantumCircuit(num_qubits)
    trotter_step.rx(theta, range(num_qubits))
    for layer in layers:
        trotter_step.compose(layer, range(num_qubits), inplace=True)

    # 指定した数のトロッター・ステップの回路を構成する
    circuit = QuantumCircuit(num_qubits)
    for _ in range(num_steps):
        circuit.rx(theta, range(num_qubits))
        for layer in layers:
            circuit.compose(layer, range(num_qubits), inplace=True)

    circuit.metadata["physical_qubits"] = tuple(qubits)
    return circuit


"""Result visualization functions"""


def plot_trotter_results(
    pub_result: PubResult,
    angles: Sequence[float],
    plot_noise_factors: Sequence[float] | None = None,
    plot_extrapolator: Sequence[str] | None = None,
    exact: np.ndarray = None,
    close: bool = True,
):
    """Plot average magnetization from ZNE result data.
    Args:
        pub_result: The Estimator PubResult for the PEA experiment.
        angles: The Rx angle values for the experiment.
        plot_raw: If provided plot the unextrapolated data for the noise factors.
        plot_extrapolator: If provided plot all extrapolators, if False only plot
            the Automatic method.
        exact: Optional, the exact values to include in the plot. Should be a 1D
            array-like where the values represent exact magnetization.
        close: Close the Matplotlib figure before returning.
    Returns:
        The figure.
    """
    data = pub_result.data

    evs = data.evs
    num_qubits = evs.shape[0]
    num_params = evs.shape[1]
    angles = np.asarray(angles).ravel()
    if angles.shape != (num_params,):
        raise ValueError(
            f"Incorrect number of angles for input data {angles.size} != {num_params}"
        )

    # 量子ビットの平均磁化とその標準誤差を求める
    x_vals = angles / np.pi
    y_vals = np.mean(evs, axis=0)
    y_errs = np.std(evs, axis=0) / np.sqrt(num_qubits)

    fig, _ = plt.subplots(1, 1)

    # 自動選択された手法の結果をプロットする
    plt.errorbar(x_vals, y_vals, y_errs, fmt="o-", label="ZNE (automatic)")

    # 各外挿モデルの結果をプロットする
    if plot_extrapolator:
        y_vals_extrap = np.mean(data.evs_extrapolated, axis=0)
        y_errs_extrap = np.std(data.evs_extrapolated, axis=0) / np.sqrt(
            num_qubits
        )
        for i, extrap in enumerate(plot_extrapolator):
            plt.errorbar(
                x_vals,
                y_vals_extrap[:, i, 0],
                y_errs_extrap[:, i, 0],
                fmt="s-.",
                alpha=0.5,
                label=f"ZNE ({extrap})",
            )

    # 生の結果をプロットする
    if plot_noise_factors:
        y_vals_raw = np.mean(data.evs_noise_factors, axis=0)
        y_errs_raw = np.std(data.evs_noise_factors, axis=0) / np.sqrt(
            num_qubits
        )
        for i, nf in enumerate(plot_noise_factors):
            plt.errorbar(
                x_vals,
                y_vals_raw[:, i],
                y_errs_raw[:, i],
                fmt="d:",
                alpha=0.5,
                label=f"Raw (nf={nf:.1f})",
            )

    # 厳密なデータをプロットする
    if exact is not None:
        plt.plot(x_vals, exact, "--", color="black", alpha=0.5, label="Exact")

    plt.ylim(-0.1, 1.2)
    plt.xlabel("θ/π")
    plt.ylabel(r"$\overline{\langle Z \rangle}$")
    plt.legend()
    plt.title(
        f"Error Mitigated Average Magnetization for Rx(θ) [{num_qubits}-qubit]"
    )
    if close:
        plt.close(fig)
    return fig


def plot_qubit_zne_data(
    pub_result: PubResult,
    angles: Sequence[float],
    qubit: int,
    noise_factors: Sequence[float],
    extrapolator: Sequence[str] | None = None,
    extrapolated_noise_factors: Sequence[float] | None = None,
    num_cols: int | None = None,
    close: bool = True,
):
    """Plot ZNE extrapolation data for specific virtual qubit
    Args:
        pub_result: The Estimator PubResult for the PEA experiment.
        angles: The Rx theta angles used for the experiment.
        qubit: The virtual qubit index to plot.
        noise_factors: the raw noise factors.
        extrapolator: The extrapolator metadata for multiple extrapolators.
        extrapolated_noise_factors: The noise factors used for extrapolation.
        num_cols: The number of columns for the generated subplots.
        close: Close the Matplotlib figure before returning.
    Returns:
        The Matplotlib figure.
    """
    data = pub_result.data

    evs_auto = data.evs[qubit]
    stds_auto = data.stds[qubit]
    evs_extrap = data.evs_extrapolated[qubit]
    stds_extrap = data.stds_extrapolated[qubit]
    evs_raw = data.evs_noise_factors[qubit]
    stds_raw = data.stds_noise_factors[qubit]

    num_params = evs_auto.shape[0]
    angles = np.asarray(angles).ravel()
    if angles.shape != (num_params,):
        raise ValueError(
            f"Incorrect number of angles for input data {angles.size} != {num_params}"
        )

    # 正方形に近いサブプロットを作成する
    num_cols = num_cols or int(np.ceil(np.sqrt(num_params)))
    num_rows = int(np.ceil(num_params / num_cols))
    fig, axes = plt.subplots(
        num_rows, num_cols, sharex=True, sharey=True, figsize=(12, 5)
    )
    fig.suptitle(f"ZNE data for virtual qubit {qubit}")

    for pidx, ax in zip(range(num_params), axes.flat):
        # 自動選択された外挿結果をプロットする
        ax.errorbar(
            0,
            evs_auto[pidx],
            stds_auto[pidx],
            fmt="o",
            label="PEA (automatic)",
        )

        # 外挿モデルをプロットする
        if (
            extrapolator is not None
            and extrapolated_noise_factors is not None
        ):
            for i, method in enumerate(extrapolator):
                ax.errorbar(
                    extrapolated_noise_factors,
                    evs_extrap[pidx, i],
                    stds_extrap[pidx, i],
                    fmt="-",
                    alpha=0.5,
                    label=f"PEA ({method})",
                )

        # 生のデータをプロットする
        ax.errorbar(
            noise_factors, evs_raw[pidx], stds_raw[pidx], fmt="d", label="Raw"
        )

        ax.set_yticks([0, 0.5, 1, 1.5, 2])
        ax.set_ylim(0, max(1, 1.1 * max(evs_auto)))

        ax.set_xticks([0, *noise_factors])
        ax.set_title(f"θ/π = {angles[pidx]/np.pi:.2f}")
        if pidx == 0:
            ax.set_ylabel(r"$\langle Z_{" + str(qubit) + r"} \rangle$")
        if pidx == num_params - 1:
            ax.set_xlabel("Noise Factor")
            ax.legend()
    plt.tight_layout()
    if close:
        plt.close(fig)
    return fig

## 小規模なシミュレーターでの例

Runtime のエラー緩和はシミュレーターではサポートされていないため、このステップは省略します。

## 大規模なハードウェアでの例

### ステップ 1: 古典的な入力を量子問題へマッピングする

#### パラメーター化されたイジング模型の回路を作成する

##### バックエンドを決める

まず、実行するバックエンドを選びます。このデモは 127 量子ビットのバックエンドで実行しますが、利用可能な任意のバックエンドに変更できます。

In [2]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend

<IBMBackend('ibm_fez')>

##### エンタングリング層のカップリングを定義する

トロッター化したイジング・シミュレーションを実装するために、デバイスに対して 2 量子ビット・ゲートのカップリングからなる 3 つの層を定義し、各トロッター・ステップで繰り返します。これらは、エラー緩和を実装するためにノイズを学習する必要がある、ツワーリングした 3 つの層を定義します。

In [3]:
layer_couplings = construct_layer_couplings(backend)
for i, layer in enumerate(layer_couplings):
    print(f"層 {i}:\n{layer}\n")

Layer 0:
[(2, 3), (4, 5), (6, 7), (8, 9), (10, 11), (12, 13), (14, 15), (16, 23), (18, 31), (19, 35), (20, 21), (25, 37), (26, 27), (28, 29), (33, 39), (36, 41), (38, 49), (42, 43), (45, 46), (47, 57), (51, 52), (53, 54), (56, 63), (58, 71), (59, 75), (61, 62), (64, 65), (66, 67), (68, 69), (72, 73), (76, 81), (79, 93), (82, 83), (84, 85), (86, 87), (88, 89), (91, 98), (94, 95), (97, 107), (99, 115), (100, 101), (102, 103), (105, 117), (108, 109), (110, 111), (113, 114), (116, 121), (118, 129), (123, 136), (124, 125), (126, 127), (130, 131), (132, 133), (135, 139), (138, 151), (142, 143), (144, 145), (146, 147), (152, 153), (154, 155)]

Layer 1:
[(0, 1), (3, 16), (5, 6), (7, 8), (11, 18), (13, 14), (17, 27), (21, 22), (23, 24), (25, 26), (29, 38), (30, 31), (32, 33), (34, 35), (39, 53), (41, 42), (43, 56), (44, 45), (47, 48), (49, 50), (51, 58), (54, 55), (57, 67), (60, 61), (62, 63), (65, 66), (69, 78), (70, 71), (73, 79), (74, 75), (77, 85), (80, 81), (83, 84), (87, 97), (89, 90), (9

##### 不良な量子ビットを取り除く

バックエンドのカップリング・マップを見て、エラーの大きいカップリングにつながっている量子ビットがないかを確認します。このような「不良な」量子ビットを実験から取り除きます。

In [4]:
# ゲート・エラーのマップをプロットする
# 注意: これらは時間とともに変化するため、結果は異なって見える場合がある
plot_error_map(backend)

<Image src="/docs/images/tutorials/probabilistic-error-amplification/extracted-outputs/fccef708-0.avif" alt="Output of the previous code cell" />

In [5]:
bad_qubits = {
    32,
    33,
    71,
    72,
    73,
    102,
    103,
}  # カップリング・エラーが大きい（1.00）ため取り除く量子ビット
good_qubits = list(set(range(backend.num_qubits)).difference(bad_qubits))
print("物理量子ビット:\n", good_qubits)

Physical qubits:
 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155]


##### トロッター回路のメインの生成処理

In [6]:
num_steps = 6
theta = Parameter("theta")
circuit = trotter_circuit(
    theta, layer_couplings, num_steps, qubits=good_qubits, backend=backend
)

#### 後で割り当てるパラメーター値のリストを作成する

In [7]:
num_params = 12

# Rx に対する [0, pi/2] の範囲の 12 個のパラメーター値。
# 観測量と外積でブロードキャストされるように形状を変更する
parameter_values = np.linspace(0, np.pi / 2, num_params).reshape(
    (num_params, 1)
)
num_params = parameter_values.size

### ステップ 2: 量子ハードウェアでの実行に向けて問題を最適化する

#### ISA 回路

ハードウェアで回路を実行する前に、ハードウェアでの実行に向けて回路を最適化します。このプロセスには、次のステップが含まれます。

* 回路の仮想量子ビットをハードウェアの物理量子ビットに対応付ける量子ビット・レイアウトを選ぶ。
* 接続されていない量子ビット間の相互作用をルーティングするために、必要に応じて SWAP ゲートを挿入する。
* 回路内のゲートを、ハードウェアで直接実行できる [命令セット・アーキテクチャー（ISA）](/docs/guides/transpile#instruction-set-architecture) の命令に変換する。
* 回路の深さとゲート数を最小化するために回路最適化を行う。

Qiskit に組み込まれているトランスパイラーはこれらのステップをすべて実行できますが、このチュートリアルでは、ユーティリティー・スケールのトロッター回路をゼロから構築する方法を示します。良好な物理量子ビットを選び、選んだ量子ビットのうち接続された量子ビットのペアに対してエンタングリング層を定義します。それでも、回路内の ISA でないゲートを変換し、トランスパイラーが提供する回路最適化を活用する必要はあります。

パス・マネージャーを作成し、それを回路に対して実行することで、選んだバックエンド向けに回路をトランスパイルします。また、回路の初期レイアウトを、すでに選んだ `good_qubits` に固定します。パス・マネージャーを作成する簡単な方法は、[`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager) 関数を使うことです。パス・マネージャーを使ったトランスパイルの詳しい説明は、[パス・マネージャーを使ったトランスパイル](/docs/guides/transpile-with-pass-managers) を参照してください。

In [8]:
pm = generate_preset_pass_manager(
    backend=backend,
    initial_layout=good_qubits,
    layout_method="trivial",
    optimization_level=1,
)

isa_circuit = pm.run(circuit)

#### ISA 観測量

次に、必要な数の $\langle I \rangle$ 項で埋めることで、各仮想量子ビットに対する重み 1 の $\langle Z \rangle$ 観測量をすべて作成します。

In [9]:
observables = []
num_qubits = len(good_qubits)
for q in range(num_qubits):
    observables.append(
        SparsePauliOp("I" * (num_qubits - q - 1) + "Z" + "I" * q)
    )

トランスパイルのプロセスでは、回路の仮想量子ビットがハードウェアの物理量子ビットに対応付けられています。量子ビット・レイアウトの情報は、トランスパイル済み回路の `layout` 属性に格納されています。観測量も仮想量子ビットに基づいて定義されているため、このレイアウトを観測量にも適用する必要があります。これは `SparsePauliOp` の `apply_layout` メソッドを使って行います。

次のコードブロックでは、各観測量がリストで囲まれていることに注意してください。これは、各量子ビットの観測量が各 theta の値に対して測定されるように、パラメーター値と *ブロードキャスト* するためです。プリミティブのブロードキャストのルールについては、[プリミティブのドキュメント](/docs/guides/primitives) を参照してください。

In [10]:
isa_observables = [
    [obs.apply_layout(layout=isa_circuit.layout)] for obs in observables
]

### ステップ 3: Qiskit プリミティブを使って実行する

In [11]:
pub = (isa_circuit, isa_observables, parameter_values)

#### Estimator のオプションを設定する

次に、エラー緩和の実験を実行するために必要な `Estimator` のオプションを設定します。これには、エンタングリング層のノイズ学習のオプションと、ZNE の外挿のオプションが含まれます。

次の設定を使います。

In [12]:
# 実験のオプション
num_randomizations = 700
num_randomizations_learning = 40
max_batch_circuits = 3 * num_params
shots_per_randomization = 64
learning_pair_depths = [0, 1, 2, 4, 6, 12, 24]
noise_factors = [1, 1.3, 1.6]
extrapolated_noise_factors = np.linspace(0, max(noise_factors), 20)

# 基本となるオプションの形式
options = {
    # ZNE のための組み込みのレジリエンス設定
    "resilience": {
        "measure_mitigation": True,
        "zne_mitigation": True,
        # TREX のノイズ学習の設定
        "measure_noise_learning": {
            "num_randomizations": num_randomizations_learning,
            "shots_per_randomization": 1024,
        },
        # PEA のノイズモデルの設定
        "layer_noise_learning": {
            "max_layers_to_learn": 3,
            "layer_pair_depths": learning_pair_depths,
            "shots_per_randomization": shots_per_randomization,
            "num_randomizations": num_randomizations_learning,
        },
        "zne": {
            "amplifier": "pea",
            "noise_factors": noise_factors,
            "extrapolator": ("exponential", "linear"),
            "extrapolated_noise_factors": extrapolated_noise_factors.tolist(),
        },
    },
    # ランダム化の設定
    "twirling": {
        "num_randomizations": num_randomizations,
        "shots_per_randomization": shots_per_randomization,
        "strategy": "active-circuit",
    },
    # オプションの動的デカップリング（DD）
    "dynamical_decoupling": {"enable": True, "sequence_type": "XY4"},
    # ジョブのタグ
    "environment": {"job_tags": ["TUT_PEA"]},
}

##### ZNE のオプションの説明

以下では、実験的ブランチで追加されたオプションについて詳しく説明します。これらのオプションと名前は確定したものではなく、正式リリースまでにすべて変更される可能性があることに注意してください。

* **amplifier**: 目的のノイズ係数までノイズを増幅する際に使う方法。
  指定できる値は、2 量子ビットの基底ゲートを繰り返すことで増幅する `"gate_folding"` と、
  ツワーリングした 2 量子ビットの基底ゲートの層について、パウリ・ツワーリングしたノイズモデルを学習した後、確率的なサンプリングによって増幅する
  `"pea"` です。その他のオプションとして `"gate_folding_front"` と `"gate_folding_back"` があり、[API ドキュメント](/docs/api/qiskit-ibm-runtime/options-models-zne-options#amplifier) で説明されています。
* **extrapolated\_noise\_factors**: 外挿モデルを評価する 1 つ以上のノイズ係数の値を指定します。
  値の列を指定した場合、返される結果は、指定したノイズ係数で外挿モデルを評価した配列になります。値
  0 はゼロノイズ外挿に対応します。

#### 実験を実行する

In [13]:
estimator = Estimator(mode=backend, options=options)
job = estimator.run([pub])
print(f"ジョブ ID {job.job_id()}")

Job ID d7fa8oe2cugc739qbb10


In [14]:
job.status()

'DONE'

### ステップ 4: 後処理を行い、目的の古典形式で結果を返す

実験が完了したら、結果を確認できます。生の期待値と緩和後の期待値を取得し、厳密な結果と比較します。次に、各パラメーターについて全量子ビットで平均した期待値を、緩和後（外挿後）と生のものの両方でプロットします。最後に、選んだ個々の量子ビットについて期待値をプロットします。

In [15]:
primitive_result = job.result()

#### 結果の一般的な形状とメタデータ

`PrimitiveResult` オブジェクトには、`PubResult` という名前のリストのような構造が含まれています。Estimator に投入する PUB は 1 つだけなので、`PrimitiveResult` には `PubResult` オブジェクトが 1 つだけ含まれます。

PUB（primitive unified bloc）の結果の期待値と標準誤差は配列です。ZNE を使った Estimator のジョブでは、`PubResult` の `DataBin` コンテナーに、期待値と標準誤差のデータ・フィールドがいくつか用意されています。ここでは期待値のデータ・フィールドについて簡単に説明します（標準誤差（`stds`）にも同様のデータ・フィールドがあります）。

1. `pub_result.data.evs`: ノイズゼロに対応する期待値（ヒューリスティックに最適と判断された外挿に基づく）。
   * 第 1 軸は、観測量 $\langle Z_i\rangle$ に対する仮想量子ビットのインデックスです（$124$ 個の仮想量子ビット／観測量）
   * 第 2 軸は、$\theta$ のパラメーター値のインデックスです（$12$ 個のパラメーター値）
2. `pub_result.data.evs_extrapolated`: 各外挿モデルについて、外挿したノイズ係数での期待値。この配列には 2 つの軸が追加されています。
   * 第 3 軸は、外挿手法のインデックスです（`exponential` と `linear` の $2$ つの外挿モデル）
   * 最後の軸は、`extrapolated_noise_factors` のインデックスです（オプションで指定した $20$ 個の外挿点）
3. `pub_result.data.evs_noise_factors`: 各ノイズ係数での生の期待値。
   * 第 3 軸は、生の `noise_factors` のインデックスです（$3$ 個の係数）

In [16]:
pub_result = primitive_result[0]

print(
    f"{pub_result.data.evs.shape=}\n"
    f"{pub_result.data.evs_extrapolated.shape=}\n"
    f"{pub_result.data.evs_noise_factors.shape=}\n"
)

pub_result.data.evs.shape=(149, 12)
pub_result.data.evs_extrapolated.shape=(149, 12, 2, 20)
pub_result.data.evs_noise_factors.shape=(149, 12, 3)



`PrimitiveResult` では、いくつかのメタデータ・フィールドも利用できます。メタデータには次のものが含まれます。

* `resilience/zne/noise_factors`: 生のノイズ係数
* `resilience/zne/extrapolator`: 各結果に使われた外挿モデル

In [17]:
primitive_result.metadata

{'dynamical_decoupling': {'enable': True,
  'sequence_type': 'XY4',
  'extra_slack_distribution': 'middle',
  'scheduling_method': 'alap'},
 'twirling': {'enable_gates': True,
  'enable_measure': True,
  'num_randomizations': 700,
  'shots_per_randomization': 64,
  'interleave_randomizations': True,
  'strategy': 'active-circuit'},
 'resilience': {'measure_mitigation': True,
  'zne_mitigation': True,
  'pec_mitigation': False,
  'zne': {'noise_factors': [1.0, 1.3, 1.6],
   'extrapolator': ['exponential', 'linear'],
   'extrapolated_noise_factors': [0.0,
    0.08421052631578947,
    0.16842105263157894,
    0.25263157894736843,
    0.3368421052631579,
    0.42105263157894735,
    0.5052631578947369,
    0.5894736842105263,
    0.6736842105263158,
    0.7578947368421053,
    0.8421052631578947,
    0.9263157894736842,
    1.0105263157894737,
    1.0947368421052632,
    1.1789473684210525,
    1.263157894736842,
    1.3473684210526315,
    1.431578947368421,
    1.5157894736842106,
    1.

`PubResult` オブジェクトには、エラー緩和に使われた学習済みノイズモデルに関する、追加のレジリエンス・メタデータがあります。

In [18]:
# 学習した層のノイズのメタデータを表示する
for field, value in pub_result.metadata["resilience"]["layer_noise"].items():
    print(f"{field}: {value}")

noise_overhead: 9.2584227461744e+229
total_mitigated_layers: 18
unique_mitigated_layers: 3
unique_mitigated_layers_noise_overhead: [2.0713004613510885e+36, 10.600275591731494, 9.687147432958504]


In [ ]:
# 元の参考文献で説明されている方法で計算した厳密なデータ
# Y. Kim et al. "Evidence for the utility of quantum computing before fault tolerance" (Nature 618,
# 500–505 (2023))。簡潔さのため、ここでは値を直接使う
exact_data = np.array(
    [
        1,
        0.9899,
        0.9531,
        0.8809,
        0.7536,
        0.5677,
        0.3545,
        0.1607,
        0.0539,
        0.0103,
        0.0012,
        0.0,
    ]
)

### トロッター・シミュレーションの結果をプロットする

次のコードでは、生の実験結果と緩和後の実験結果を厳密解と比較するプロットを作成します。

In [20]:
zne_metadata = primitive_result.metadata["resilience"]["zne"]
# トロッター・シミュレーションの結果をプロットする
fig = plot_trotter_results(
    pub_result,
    parameter_values,
    plot_extrapolator=zne_metadata["extrapolator"],
    plot_noise_factors=zne_metadata["noise_factors"],
    exact=exact_data,
)
display(fig)

<Image src="/docs/images/tutorials/probabilistic-error-amplification/extracted-outputs/e466736a-0.avif" alt="Output of the previous code cell" />

ノイズを含む値（ノイズ係数 `nf=1.0`）は厳密な値から大きくずれていますが、緩和後の値は厳密な値に近く、PEA に基づくエラー緩和手法の有用性が示されています。

### 個々の量子ビットの外挿結果をプロットする

最後に、次のコードでは、特定の量子ビットについて、theta のさまざまな値に対する外挿曲線を示すプロットを作成します。

In [21]:
virtual_qubit = 1
plot_qubit_zne_data(
    pub_result=pub_result,
    angles=parameter_values,
    qubit=virtual_qubit,
    noise_factors=zne_metadata["noise_factors"],
    extrapolator=zne_metadata["extrapolator"],
    extrapolated_noise_factors=zne_metadata["extrapolated_noise_factors"],
)

<Image src="/docs/images/tutorials/probabilistic-error-amplification/extracted-outputs/bea9695a-0.avif" alt="Output of the previous code cell" />

## 次のステップ

<Admonition type="tip" title="おすすめ">
  この内容に興味を持たれた方には、次の資料もお勧めします。

  * エラー緩和手法の組み合わせに焦点を当てた [チュートリアル](/docs/tutorials/combine-error-mitigation-techniques)
  * Qiskit で利用できるエラー緩和手法に関する詳しい [ドキュメント](/docs/guides/error-mitigation-and-suppression-techniques)
  * ユーティリティー・スケールの実験を扱うその他のレッスン: [Utility II](/learning/courses/utility-scale-quantum-computing/utility-ii) と [Utility III](/learning/courses/utility-scale-quantum-computing/utility-iii)
</Admonition>

© IBM Corp., 2017-2026